# Attention Sink Research — Phase 3
## Notebook 04A · Phase 3.1 — Pre-Softmax Attention Capture

**Runtime:** GPU (free **T4** is enough).
**Depends on:** `phase3_utils.py` in your Drive project folder. Upload it once — it persists.
**Independent of:** Phase 1 / Phase 2 outputs. This notebook runs the model itself.

---

### What this notebook does

Phases 1 and 2 measured attention **probabilities**. Those are the output of a
softmax, and the softmax destroys information: `dp/dl = p(1-p)`, so once a head
sits at `p ≈ 0.95` a difference of several *nats* in the underlying logits gets
compressed into a few thousandths of probability. Phase 3.1 goes one step
upstream and records the computation **before** the softmax.

For every layer, head, query position and key position:

| quantity | shape | notes |
|---|---|---|
| pre-softmax logits `QK^T/√d` | `[L, H, T, T]` | **raw and unmasked** — the causal mask is applied downstream so the stored tensor stays a clean mathematical object |
| post-softmax probabilities | `[L, H, T, T]` | exactly what the model used |
| query vectors | `[L, H, T, D]` | post-`q_norm`, post-RoPE |
| key vectors | `[L, G, T, D]` | post-`k_norm`, post-RoPE, **pre-`repeat_kv`** — one per KV head |
| query / key norms | `[L, H, T]` / `[L, G, T]` | computed in fp32 regardless of model dtype |

And per query token, in **nats**: BOS probability, BOS logit, max competing
logit, log-sum-exp of competitors, both margins, and attention entropy.

### Why this is not a forward hook

`Qwen3Attention.forward` dispatches to an *attention-interface function*:

```python
attention_interface = eager_attention_forward
if config._attn_implementation != "eager":
    attention_interface = ALL_ATTENTION_FUNCTIONS[config._attn_implementation]
```

Pre-softmax logits exist only **inside** that function — they are never a module
output, so no `nn.Module` forward hook can reach them. `phase3_utils` registers a
byte-for-byte replica of `eager_attention_forward` under a private random name
(plus the matching *mask* function, or `create_causal_mask` raises `KeyError`),
points the config at it, and restores the original on exit — including on
exception. Part 2 proves the LM-head logits come out **bit-identical** with and
without capture.

### Prompts are a controlled variable

This notebook draws **every** prompt from the frozen 50-pair En–Vi benchmark
(`benchmark_en_vi.csv`) that Phase 2 ran on — the one marked *"FROZEN for Phases
2–4 to maximise reproducibility"*. It does **not** define prompts of its own, and
it **refuses to run** if the benchmark is missing rather than falling back to a
substitute corpus. A fallback would produce plausible-looking figures that are
silently incomparable to Phase 2, which is the exact failure this control exists
to prevent.

The SHA-256 of the normalised pair list is recorded in the manifest, so
benchmark drift between phases is detectable after the fact.

### Hand-off

Everything Phase 3.2 (shared-KV specialisation) and Phase 3.3 (query-pathway
intervention) need is written to `results/phase3/experiment3.1/`. Neither should
ever re-run inference to ask a question about *observed* attention.

In [ ]:
# --- Colab environment setup -------------------------------------------------
# Detect Colab, install the versions Qwen3 needs, and confirm a GPU is attached.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, sys
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-U',
         'transformers>=4.51', 'accelerate'],
        check=True,
    )

import torch, transformers
print('In Colab      :', IN_COLAB)
print('torch         :', torch.__version__)
print('transformers  :', transformers.__version__, '(>=4.51 required for Qwen3)')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU           :', props.name, f'({props.total_memory/1024**3:.1f} GB)')
    print('bf16 native   :', torch.cuda.is_bf16_supported(), '(T4 = False)')
else:
    print('*** No GPU. Runtime > Change runtime type > Hardware accelerator > GPU (T4). ***')

In [ ]:
# --- Storage + phase3_utils bootstrap ---------------------------------------
# Same Drive project folder Phases 1 and 2 use. Colab wipes local disk on reset,
# so anything Phase 3.2 / 3.3 must read has to live on Drive.
USE_DRIVE = True   # keep consistent with Phase 1 / Phase 2

import sys
from pathlib import Path
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
elif IN_COLAB:
    BASE = Path('/content/attention_sink_project')
else:
    BASE = Path('.')
BASE.mkdir(parents=True, exist_ok=True)

RESULTS_ROOT = BASE / 'results' / 'phase3'
EXP_DIR      = RESULTS_ROOT / 'experiment3.1'
for sub in ('captures', 'metrics', 'figures', 'analysis'):
    (EXP_DIR / sub).mkdir(parents=True, exist_ok=True)

# Locate phase3_utils.py (upload it into BASE once; it persists on Drive).
for c in [BASE, Path('/content'), Path('.')]:
    if (Path(c) / 'phase3_utils.py').exists():
        sys.path.insert(0, str(c)); break
try:
    import phase3_utils as U
    print('phase3_utils loaded from', U.__file__, '| version', U.__version__)
except ModuleNotFoundError:
    raise SystemExit('Place phase3_utils.py in ' + str(BASE) + ' (or /content) and re-run this cell.')

print('Phase 3 out   :', RESULTS_ROOT)
print('Experiment    :', EXP_DIR)

---
## Part 1 · Configuration

Two settings deserve a moment rather than a default.

**`DTYPE = 'float32'`.** bfloat16 carries ~8 mantissa bits and we are comparing
margins at the 0.1-nat level — bf16 rounding is the same order as the effect
Phase 3.2 is looking for, and that noise lands directly on the effect size.
fp32 weights for Qwen3-1.7B are 6.8 GB, which fits a T4's 15 GB alongside
activations at these sequence lengths. Change it only if you are memory-bound,
and expect the validation tolerances to loosen by ~4 orders of magnitude.

**`SAVE_FULL_MATRICES = False`.** The `[L, H, T, T]` tensors are ~99% of the
payload and scale quadratically in `T` (see the estimate printed below). Phases
3.2 and 3.3 need only `q`, `k` and the derived `[L, H, T]` metrics — both of
which are always saved. The full matrices are still *computed* in memory and
still drive every metric and every validation check; they are just not written
to Drive. Set `True` if you want to re-derive metrics later without re-running
inference.

`PREPEND_BOS = True` matters more than it looks: **Qwen3's tokenizer does not add
a BOS token.** Without it, "position 0" is just whatever word happens to be
first, and you would be measuring a first-position effect rather than a BOS
effect. Both are legitimate objects of study; they are not the same object. The
metadata records `sink_is_bos` so you always know which one you got.

In [ ]:
# --- Configuration -----------------------------------------------------------
CONFIG = dict(
    MODEL_NAME   = 'Qwen/Qwen3-1.7B',

    # --- prompts: drawn from the FROZEN benchmark, never defined here --------
    PROMPT_MODE  = 'category_block',   # 'per_sentence' | 'category_block' | 'full_block'
    LANGUAGES    = ('eng', 'vie'),     # the benchmark is parallel -- keep both
    CATEGORIES   = None,               # None = all 5; or e.g. ['news', 'technical']
    MAX_SEQUENCES = None,              # cap the number of captures
    STRICT_BENCHMARK = True,           # fail on any benchmark validation warning

    MAX_LEN      = 256,        # [L,H,T,T] scales as T^2 -- see the audit in Part 4
    DTYPE        = 'float32',  # 'float32' | 'bfloat16' | 'float16'
    LAYERS       = None,       # None = all 28; or e.g. [7, 14, 21, 27]
    PREPEND_BOS  = True,       # Qwen3's tokenizer does NOT add BOS on its own
    SAVE_FULL_MATRICES = False,# write [L,H,T,T] logits/probs to Drive?
    MIN_QUERY_POS = 4,         # drop early positions: competitor sets too small
    MAX_POINTS   = 40_000,     # scatter subsample for the figures
    SAVE_BUDGET_GB = 2.0,      # warn + suggest a layer subset above this
    SEED         = 0,
    RUN_SELF_TEST = True,      # Part 3: offline plumbing check, ~10 s
)

torch.manual_seed(CONFIG['SEED'])
for k, v in CONFIG.items():
    print(f'  {k:<20} {v}')

---
## Part 2 · Load the frozen benchmark

Phase 3 reuses the **same 50 sentence pairs** Phase 2 ran on — 10 each across
conversational, news, technical, academic and literary — so that the only thing
differing between Phase 2 and Phase 3 is *what is measured*, not *what it is
measured on*.

`load_benchmark` **raises** if the benchmark is absent. There is deliberately no
fallback corpus: substituting prompts would yield figures that look fine and are
not comparable to anything. It also validates the pair count, uniqueness,
categories, and the presence of Vietnamese diacritics — that last one catches the
old un-accented placeholder corpus, which tokenises very differently.

### The one construction choice

Benchmark sentences target 15–40 tokens. That is right for Phase 1/2 sink
scores, but too short for parts of Phase 3.1: the competitor set at query
position `t` has only `t` members, so `logsumexp` over competitors cannot show
its `log t` growth, and entropy is capped at `log(t+1)`.

`PROMPT_MODE = 'category_block'` therefore joins the 10 sentences of a category
in **id order** into one sequence — the same construction Phase 2's 2C used to
build its length ladder from benchmark text. Content stays exactly the benchmark;
length becomes an explicit, recorded construction rather than a new uncontrolled
variable. Every capture stores the `pair_ids` it was built from, so any result
traces back to specific benchmark rows.

Set `PROMPT_MODE = 'per_sentence'` if you want the exact Phase 2 unit of
analysis and are willing to lose the position-dependent measurements.

In [ ]:
# --- Load + validate the frozen benchmark ------------------------------------
SEARCH_ROOTS = [BASE, Path('/content'), Path('.')]
bm = U.load_benchmark(SEARCH_ROOTS, strict=CONFIG['STRICT_BENCHMARK'])

print('benchmark   :', bm['path'])
print('pairs       :', len(bm['pairs']))
print('categories  :', sorted({p['category'] for p in bm['pairs']}))
print('fingerprint :', bm['fingerprint'])
if bm['warnings']:
    print('warnings    :', bm['warnings'])

prompts = U.build_prompts(
    bm['pairs'],
    mode=CONFIG['PROMPT_MODE'],
    languages=CONFIG['LANGUAGES'],
    categories=CONFIG['CATEGORIES'],
    max_sequences=CONFIG['MAX_SEQUENCES'],
)
print()
print(U.describe_prompts(prompts))
print('\nfirst 110 chars of', prompts[0].seq_id, '->', repr(prompts[0].text[:110]))

---
## Part 3 · Pipeline self-test (offline, no checkpoint)

Before spending session time downloading 1.7B parameters, verify the capture
machinery on a **randomly-initialised** small Qwen3 built straight from
`Qwen3Config` — real GQA, real QK-RMSNorm, `rope_theta=1e6`, no attention bias.
Takes a few seconds and needs no network.

Every invariant checked here is *architectural*, not weight-dependent, so
passing on a random model is exactly as informative as passing on the real
checkpoint for everything except absolute magnitudes:

| check | identity |
|---|---|
| `C1` | `softmax(masked stored logits) == stored probs` — the tensor really is pre-softmax |
| `C2` | `q @ repeat_kv(k).T * scaling == stored logits` — q/k intact, head→KV map is `h // n_rep` |
| `C2b` | a *shifted* head→KV map does **not** reproduce the logits — so C2 isn't vacuous |
| `C3` | probs normalise over causal keys; exactly zero on future keys |
| `C4` | `p_sink == sigmoid(margin_lse)` — end-to-end |
| `C5` | `‖q‖, ‖k‖ ∈ [√D·min\|γ\|, √D·max\|γ\|]` — QK-norm is active |
| `C6` | `0 ≤ H(t) ≤ log(n_valid_keys)` |

`C5` is the Qwen3-specific one. RMSNorm over `head_dim` plus norm-preserving
RoPE gives `‖k‖² = Σᵢ γᵢ² uᵢ²` with `‖u‖² = D`, so key norms **cannot** escape an
interval fixed by the learned gains — the "massive key norm at BOS" mechanism
reported for GPT-2 and Llama is structurally unavailable in Qwen3. The check
prints the max achievable spread, so when you run it on the real checkpoint in
Part 4 that number tells you directly how much room the magnitude channel has.

In [ ]:
# --- Offline plumbing check --------------------------------------------------
if CONFIG['RUN_SELF_TEST']:
    from transformers import Qwen3Config, Qwen3ForCausalLM

    tiny_cfg = Qwen3Config(
        vocab_size=256, hidden_size=64, intermediate_size=128, num_hidden_layers=3,
        num_attention_heads=8, num_key_value_heads=4, head_dim=16,
        max_position_embeddings=512, rope_theta=1_000_000.0, attention_bias=False,
        bos_token_id=1, eos_token_id=2,
    )
    torch.manual_seed(0)
    tiny = Qwen3ForCausalLM(tiny_cfg).to(torch.float32).eval()
    try:
        tiny.set_attn_implementation('eager')
    except Exception:
        tiny.config._attn_implementation = 'eager'
    with torch.no_grad():   # perturb RMSNorm gains so the C5 interval is non-trivial
        for layer in tiny.model.layers:
            layer.self_attn.q_norm.weight.normal_(1.0, 0.25).abs_().clamp_(min=0.3)
            layer.self_attn.k_norm.weight.normal_(1.0, 0.25).abs_().clamp_(min=0.3)

    ids = torch.randint(3, 256, (1, 48), generator=torch.Generator().manual_seed(0))
    ids[0, 0] = 1

    with torch.no_grad():
        ref = tiny(ids, use_cache=False, output_attentions=True)

    with U.capture_attention(tiny, U.CaptureConfig()) as rec:
        with torch.no_grad():
            got = tiny(ids, use_cache=False)
    cap  = rec.finalize(input_ids=ids, model=tiny, seq_id='selftest')
    qm   = U.compute_query_metrics(cap)

    # non-invasiveness: the model's own output must be untouched
    assert torch.equal(ref.logits, got.logits), 'capture changed the LM head logits!'
    assert tiny.config._attn_implementation == 'eager', 'attn implementation not restored'
    # our probs must equal stock eager attention
    worst = max((a[0].double() - cap.probs[i].double()).abs().max().item()
                for i, a in enumerate(ref.attentions))
    print(f'LM logits bit-identical with/without capture : True')
    print(f'attn implementation restored                 : True')
    print(f'max |our probs - HF output_attentions|       : {worst:.2e}\n')

    report = U.validate_capture(cap, qm, model=tiny)
    print(report)
    report.raise_if_failed()
    print(f"\nhead -> KV group map (n_rep={cap.n_rep}): {cap.kv_group_of_head.tolist()}")
    del tiny, cap, qm
else:
    print('self-test skipped (RUN_SELF_TEST=False)')

---
## Part 4 · Load Qwen3-1.7B

`attn_implementation='eager'` at load time. SDPA and flash never materialise an
attention matrix, so there would be nothing to intercept; the capture swaps the
implementation anyway, but loading eager keeps `output_attentions` available for
cross-checking and avoids a silent fallback.

In [ ]:
# --- Load model + tokenizer --------------------------------------------------
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

DTYPE = {'float32': torch.float32, 'bfloat16': torch.bfloat16,
         'float16': torch.float16}[CONFIG['DTYPE']]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if CONFIG['DTYPE'] != 'float32':
    print(f"[warn] DTYPE={CONFIG['DTYPE']}: margins are compared at the 0.1-nat level "
          f"and reduced precision puts rounding noise at the same order. "
          f"Validation tolerances loosen accordingly.\n")

t0 = time.time()
tok = AutoTokenizer.from_pretrained(CONFIG['MODEL_NAME'])
try:                                    # transformers renamed torch_dtype -> dtype
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG['MODEL_NAME'], dtype=DTYPE, attn_implementation='eager')
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG['MODEL_NAME'], torch_dtype=DTYPE, attn_implementation='eager')
model = model.to(DEVICE).eval()

c = model.config
n_par = sum(p.numel() for p in model.parameters())
print(f"loaded in {time.time()-t0:.1f}s on {DEVICE} / {CONFIG['DTYPE']}")
print(f"  params      : {n_par/1e9:.2f} B  ({n_par*DTYPE.itemsize/1024**3:.1f} GB)")
print(f"  layers      : {c.num_hidden_layers}")
print(f"  heads / KV  : {c.num_attention_heads} / {c.num_key_value_heads} "
      f"(n_rep = {c.num_attention_heads // c.num_key_value_heads})")
print(f"  head_dim    : {c.head_dim}   rope_theta: {c.rope_theta:,.0f}")
print(f"  attn impl   : {c._attn_implementation}")
print(f"  QK-norm     : q_norm={hasattr(model.model.layers[0].self_attn,'q_norm')} "
      f"k_norm={hasattr(model.model.layers[0].self_attn,'k_norm')}")
print(f"  bos_token_id: {c.bos_token_id}  "
      f"(tokenizer adds BOS: {tok('hi')['input_ids'][:1] == [c.bos_token_id]})")

---
## Part 5 · Token-length audit and storage projection

Now that a real tokenizer exists, check what the benchmark actually costs before
committing. Two things to look at.

**Token lengths.** Vietnamese with full diacritics tokenises into noticeably more
subwords than the parallel English, so the En and Vi blocks of the same category
will not have equal `T`. That is expected and is not a confound — every metric
here is computed per query position and carries `n_valid_keys` — but it does mean
the two languages sit at different points on the position axis, so compare them
at matched `query_pos` rather than in aggregate.

**Storage.** The `[L, H, T, T]` matrices scale quadratically in `T` and linearly
in the number of sequences. If the projection exceeds `SAVE_BUDGET_GB` the cell
prints a suggested layer subset. Subsetting layers is scientifically defensible
here rather than a pure storage hack: Phase 1 established that sinks emerge
progressively with depth, so the later layers are where the signal is.

In [ ]:
# --- What this run will actually cost ----------------------------------------
print(U.describe_prompts(prompts, tokenizer=tok, max_len=CONFIG['MAX_LEN']))

tok_lens = [len(tok(p.text, add_special_tokens=False)['input_ids']) for p in prompts]
T_eff = [min(n + (1 if CONFIG['PREPEND_BOS'] else 0), CONFIG['MAX_LEN']) for n in tok_lens]

_cfg_est = U.CaptureConfig(layers=CONFIG['LAYERS'])
lean_total = full_total = 0
for T in T_eff:
    e = U.estimate_capture_bytes(c.num_hidden_layers, c.num_attention_heads,
                                 c.num_key_value_heads, c.head_dim, T, _cfg_est)
    full_total += e['total']
    lean_total += e['total'] - e.get('logits', 0) - e.get('probs', 0)
on_disk = full_total if CONFIG['SAVE_FULL_MATRICES'] else lean_total

print(f"\npeak in-memory (largest sequence, T={max(T_eff)}): "
      f"{U.human_bytes(max(U.estimate_capture_bytes(c.num_hidden_layers, c.num_attention_heads, c.num_key_value_heads, c.head_dim, T, _cfg_est)['total'] for T in T_eff))}")
print(f"written to Drive across {len(prompts)} sequences: {U.human_bytes(on_disk)}"
      f"  (SAVE_FULL_MATRICES={CONFIG['SAVE_FULL_MATRICES']})")

budget = CONFIG['SAVE_BUDGET_GB'] * 1024**3
if on_disk > budget:
    keep = max(4, int(c.num_hidden_layers * budget / on_disk))
    step = max(1, c.num_hidden_layers // keep)
    sugg = list(range(c.num_hidden_layers - 1, -1, -step))[::-1]
    print(f"\n[!] over the {CONFIG['SAVE_BUDGET_GB']} GB budget. Options:")
    print(f"    LAYERS = {sugg}   (deeper layers: Phase 1 showed sinks strengthen with depth)")
    print(f"    or lower MAX_LEN, or set LANGUAGES = ('eng',)")
else:
    print(f"within the {CONFIG['SAVE_BUDGET_GB']} GB budget.")

---
## Part 6 · Capture, validate, persist

One full-sequence forward pass per prompt, `use_cache=False`, batch size 1.
Batching is deliberately rejected by the recorder rather than guessed at:
padding plus left/right alignment would silently corrupt the causal-mask
bookkeeping, and every number here depends on that bookkeeping being exact.

All seven validation checks run **per prompt** and the loop aborts on failure —
a bad capture should stop the notebook, not quietly reach the figures.

In [ ]:
# --- Capture loop ------------------------------------------------------------
cap_cfg = U.CaptureConfig(
    store_logits=True, store_probs=True, store_queries=True, store_keys=True,
    store_norms=True, layers=CONFIG['LAYERS'],
    store_dtype=torch.float32,   # always fp32 on disk, whatever the model dtype
    store_device='cpu', sink_position=0,
)
DROP = () if CONFIG['SAVE_FULL_MATRICES'] else ('logits', 'probs')

tables, manifest_seqs, val_summary = [], [], {}
seq_lookup = {}   # seq index -> category/language/pair_ids, for the pooled table

for i, P in enumerate(prompts):
    seq_id = P.seq_id
    ids = tok(P.text, return_tensors='pt', add_special_tokens=False)['input_ids']
    ids = ids[:, : CONFIG['MAX_LEN'] - (1 if CONFIG['PREPEND_BOS'] else 0)]
    if CONFIG['PREPEND_BOS'] and c.bos_token_id is not None:
        ids = torch.cat([torch.tensor([[int(c.bos_token_id)]], dtype=ids.dtype), ids], 1)
    ids = ids.to(DEVICE)

    t1 = time.time()
    with U.capture_attention(model, cap_cfg) as rec:
        with torch.no_grad():
            model(ids, use_cache=False)
    cap = rec.finalize(input_ids=ids, model=model, seq_id=seq_id, tokenizer=tok,
                       extra_meta={**P.as_meta(), 'prompt_index': i,
                                   'prepend_bos': CONFIG['PREPEND_BOS'],
                                   'benchmark_fingerprint': bm['fingerprint']})
    qm = U.compute_query_metrics(cap)

    report = U.validate_capture(cap, qm, model=model)
    print(f"\n[{seq_id}] {P.language}/{P.category:<15} T={cap.seq_len:<4} "
          f"pairs={P.pair_ids[0]}-{P.pair_ids[-1]}  ({time.time()-t1:.1f}s)")
    print(report)
    val_summary[seq_id] = {ck.name: ck.passed for ck in report.checks}
    report.raise_if_failed()

    # QueryMetrics first -- dropping the matrices makes them unrecoverable
    U.save_query_metrics(qm, EXP_DIR / 'metrics' / f'{seq_id}_qm.pt')
    U.save_capture(cap, EXP_DIR / 'captures' / f'{seq_id}.pt', drop=DROP)

    tbl = U.build_metric_table(qm, seq_index=i, min_query_pos=CONFIG['MIN_QUERY_POS'])
    tbl.meta.update({'category': P.category, 'language': P.language})
    tables.append(tbl)
    seq_lookup[i] = dict(seq_id=seq_id, category=P.category, language=P.language,
                         pair_ids=P.pair_ids, seq_len=cap.seq_len)
    manifest_seqs.append(dict(seq_id=seq_id, category=P.category, language=P.language,
                              pair_ids=P.pair_ids, seq_len=cap.seq_len,
                              rows=len(tbl), sink_token_id=cap.meta['sink_token_id'],
                              sink_is_bos=cap.meta['sink_is_bos']))
    del cap, qm
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print('\nall sequences captured and validated')

---
## Part 7 · Pooled metric table

`MetricTable` is the hand-off format: one tidy row per
`(seq, layer, head, kv_group, query_pos)`. Phase 3.2 groups by `kv_group`;
Phase 3.3 joins interventions on `(seq, layer, head, query_pos)`.

When you read the summary, compare heads on **`margin_lse`, not `bos_prob`**.
Above `p ≈ 0.95` the probability scale compresses several nats into thousandths,
and a 3.2 analysis run on probabilities will report "no within-group
specialisation" as a pure saturation artefact.

In [ ]:
# --- Pool, save, inspect -----------------------------------------------------
pooled = U.MetricTable.concat(tables)
pooled.meta.update({'model_name': CONFIG['MODEL_NAME'], 'sink_position': 0,
                    'prepend_bos': CONFIG['PREPEND_BOS'], 'units': 'nats',
                    'prompt_mode': CONFIG['PROMPT_MODE'],
                    'benchmark_fingerprint': bm['fingerprint'],
                    'benchmark_path': str(bm['path']),
                    'sequences': seq_lookup})   # seq index -> category / language
U.save_metric_table(pooled, EXP_DIR / 'metrics' / 'table.pt')
print(pooled.summary())

print('\nby language / category (mean LSE margin, nats):')
for i_, info in seq_lookup.items():
    sub = pooled.where(seq=i_)
    print(f"  {info['language']}/{info['category']:<16} T={info['seq_len']:<4} "
          f"margin_lse {sub['margin_lse'].mean():+.3f}   p_sink {sub['bos_prob'].mean():.4f}")

frac = (pooled['bos_prob'] > 0.9).double().mean().item()
print(f"\nfraction of (layer, head, position) with p_sink > 0.9 : {frac:.3f}")
if frac > 0.2:
    print("  -> a large share sits in the saturated regime; use prob_scale='logit' "
          "on the figures and margin_lse for any head comparison.")

---
## Part 8 · Figures

The four requested comparisons are panels (a)–(d) of `sink_panels.png`. **Read
them in order, and treat panel (c) with suspicion — deservedly.**

With the softmax taken over exactly the causally-valid keys,

$$p_{t,0} \;=\; \frac{e^{\ell_{t,0}}}{\sum_{s\le t} e^{\ell_{t,s}}}
\;=\; \frac{1}{1 + e^{\,\mathrm{LSE}_{s>0}\ell_{t,s} \,-\, \ell_{t,0}}}
\;=\; \sigma\!\left(\mathrm{margin\_lse}\right)$$

exactly. So panel (c) — LSE margin vs BOS probability — **can only ever trace the
logistic curve**. It is plotted with the analytic curve overlaid in red; any
visible deviation is a bug, not a finding. It is a correctness panel.

That makes panel (a) the interesting one. The *gap* between (a)'s scatter and
that curve is precisely the contribution of the competitor term: if `bos_logit`
alone already predicts `p_sink` tightly, the sink rides on the BOS logit; if (a)
is diffuse, the model is controlling the sink mainly by suppressing competitors
instead.

`margin_decomposition.png` splits the margin into its two additive terms against
query position on a log axis. `logsumexp` over competitors grows like `log t`
when the competitor logits are roughly exchangeable, so a sink probability that
stays flat in `t` **requires** `bos_logit` to track that growth — an active
mechanism, not a parked constant. This is the figure most likely to produce a
Phase 3 result on its own.

In [ ]:
# --- Generate figures --------------------------------------------------------
FIG = EXP_DIR / 'figures'
figs = {}
figs['sink_panels'] = U.plot_sink_panels(
    pooled, FIG / 'sink_panels.png', max_points=CONFIG['MAX_POINTS'],
    seed=CONFIG['SEED'], show=True)
figs['sink_panels_logit_y'] = U.plot_sink_panels(
    pooled, FIG / 'sink_panels_logit_y.png', max_points=CONFIG['MAX_POINTS'],
    seed=CONFIG['SEED'], prob_scale='logit', show=True)
figs['margin_decomposition'] = U.plot_margin_decomposition(
    pooled, FIG / 'margin_decomposition.png', show=True)
figs['layer_head_maps'] = U.plot_layer_head_maps(
    pooled, FIG / 'layer_head_maps.png', show=True)

for k, v in figs.items():
    print(f'  {k:<22} {v}')

---
# Extended pre-softmax analyses

The four analyses below run entirely on the pooled `MetricTable` that Part 7
saved. **None of them re-run the model.** Three are pure functions of scalars
already in the table; the competitor-structure one uses the top-k and
competitor-entropy fields, which were computed at capture time (Part 6) exactly
so this stays inference-free even with `SAVE_FULL_MATRICES=False`.

Everything is computed in float64 and is NaN-aware. Reusable tensors are written
to `results/phase3/experiment3.1/analysis/`, and the group-statistics payload is
the direct hand-off to Phase 3.2.

In [ ]:
# --- Reload the pooled table (proves the analyses are inference-free) --------
# In a fresh session you can start HERE: load table.pt and run Parts 9-12.
import phase3_utils as U
from pathlib import Path
try:
    pooled
except NameError:
    pooled = U.load_metric_table(EXP_DIR / 'metrics' / 'table.pt')
    print('reloaded pooled table from disk')

need = ['bos_logit', 'comp_lse_logit', 'comp_max_logit', 'margin_lse', 'entropy',
        'comp_top2_logit', 'comp_top3_logit', 'comp_entropy', 'comp_neff']
missing = [c for c in need if c not in pooled.names]
assert not missing, f'table is missing {missing}; re-run Part 6 with current phase3_utils'
import torch as _t
for c in ('comp_top2_logit', 'comp_neff', 'comp_entropy'):
    assert _t.isfinite(pooled[c]).any(), (
        f"'{c}' is all-NaN -- this table predates the competitor-structure fields. "
        "Re-run the capture (Part 6); no full matrices needed.")
print(f'pooled table ready: {len(pooled):,} rows, all extended columns present')

---
## Part 9 · Variance decomposition

Because `margin_lse = bos_logit − comp_lse_logit` **exactly**, the variance of
the margin splits, per layer, as

$$\mathrm{Var}(\text{margin}) = \mathrm{Var}(\text{BOS}) + \mathrm{Var}(\text{LSE}) - 2\,\mathrm{Cov}(\text{BOS}, \text{LSE}).$$

Panel (a) plots the three terms and overlays the reconstruction — they must
coincide, and the figure title reports the max identity error (expected ~1e-15).

Panel (b) is the answer to *"what drives the margin?"*. It splits `Var(margin)`
into two contributions that sum to it exactly: `Cov(margin, BOS)` and
`Cov(margin, −LSE)`. Whichever is larger is the term the model actually uses to
move the sink at that depth.

Panel (c) is `ρ(BOS logit, competitor LSE)`. Positive ρ means the model raises
the BOS logit and the competition *together* (they partly cancel in the margin);
negative ρ means it plays them against each other to amplify the margin. That
sign is a genuine mechanistic result, not a bookkeeping artefact.

In [ ]:
# --- (1) Variance decomposition ----------------------------------------------
vd = U.variance_decomposition(pooled, include_kv_group=True)
assert vd.max_identity_error < 1e-9, f'identity broken: {vd.max_identity_error}'
print(f'Var(margin) = Var(BOS)+Var(LSE)-2Cov  verified '
      f'(max error {vd.max_identity_error:.1e} nats^2)')
print(f'mean rho(BOS, LSE) across layers: {float(vd.rho[_t.isfinite(vd.rho)].mean()):+.3f}')
fig_vd = U.plot_variance_decomposition(vd, EXP_DIR / 'figures' / 'analysis_variance_decomposition.png', show=True)

---
## Part 10 · Competitor structure

For every query: the top-1/2/3 competitor logits, the competitor-only entropy
$H_{\mathrm{comp}}$, and the **effective number of competitors**
$N_{\mathrm{eff}} = e^{H_{\mathrm{comp}}}$ (the perplexity of the competitor
softmax: 1 when one token dominates, up to $t$ when all compete equally).

Panel (c) is the interpretive payoff — how $N_{\mathrm{eff}}$ co-varies with the
margin and the LSE across layers. If the margin shrinks as $N_{\mathrm{eff}}$
grows, the sink is being eroded by *breadth* of competition; if the margin
tracks the top-1 competitor instead, it is being eroded by a single rival
token.

In [ ]:
# --- (2) Competitor structure ------------------------------------------------
cs = U.competitor_structure(pooled)
print('mean N_eff by layer (first/mid/last): '
      f'{cs.mean_neff[0]:.2f} / {cs.mean_neff[len(cs.mean_neff)//2]:.2f} / {cs.mean_neff[-1]:.2f}')
print('corr(N_eff, margin) last layer: '
      f'{float(cs.r_neff_margin[-1]):+.3f}   corr(top-1, margin): {float(cs.r_top1_margin[-1]):+.3f}')
fig_cs = U.plot_competitor_structure(cs, EXP_DIR / 'figures' / 'analysis_competitor_structure.png', show=True)

---
## Part 11 · LSE decomposition — one dominant token, or many?

Rewrite the competitor log-sum-exp around its own maximum:

$$L = \ell_{\max} + \underbrace{\log \sum_i e^{\ell_i - \ell_{\max}}}_{R \ge 0}.$$

The residual $R$ has an exact reading: $R = -\log p_{\max}^{\mathrm{comp}}$, the
negative log of the single strongest competitor's share of competitor attention.
So $R \approx 0$ means **one token carries the competition**; large $R$ means
**many moderately strong competitors** do — which is precisely the question this
analysis is posed to answer.

Panel (a) stacks $\ell_{\max}$ and $R$ into the LSE. Panel (b) plots
$p_{\max}^{\mathrm{comp}} = e^{-R}$ directly: near 1 is single-token dominance,
near 0 is diffuse competition. Panel (c) cross-checks $R$ against
$N_{\mathrm{eff}}$ from Part 10 — they should move together.

This uses only `comp_max_logit` and `comp_lse_logit`, both already saved, so it
is a pure re-read.

In [ ]:
# --- (3) LSE decomposition ---------------------------------------------------
ld = U.lse_decomposition(pooled)
err = float((ld.mean_lmax + ld.mean_residual - ld.mean_lse).abs().max())
assert err < 1e-9, err
print(f'LSE = l_max + R verified (max error {err:.1e})')
print(f'p_max^comp (dominance) last layer: {float(ld.mean_pmax[-1]):.3f}  '
      f'(->1 one token, ->0 many)')
fig_ld = U.plot_lse_decomposition(ld, EXP_DIR / 'figures' / 'analysis_lse_decomposition.png', show=True)

---
## Part 12 · KV-group statistics and ICC — the Phase 3.2 hand-off

For BOS logit, competitor LSE, margin and entropy, compute per layer the
**within-KV-group variance**, the **between-KV-group variance**, and the
**intraclass correlation** ICC(1,1) over heads grouped by their shared KV head.

The unit is the head: each head's value is its mean over (position, sequence),
then the ICC is taken over heads within/between KV groups. Averaging positions
into the head mean is deliberate — it strips the strong position dependence
(the margin grows with `log t`) so the ICC reflects head *identity*.

**Reading it for Phase 3.2.** ICC is the share of head-level variance that lies
*between* groups. High ICC → heads sharing a KV head resemble each other → low
within-group specialisation. Low or negative ICC → heads within a group differ
as much as across groups → consistent with specialisation. Phase 3.2's question
("do shared-KV heads specialise?") is answered by *where ICC drops*, especially
in the later layers.

`head_means [L, H, M]` is saved alongside the ICCs, so Phase 3.2 can recompute
any grouping statistic without touching the model or the pooled table.

In [ ]:
# --- (4) Group statistics + ICC ----------------------------------------------
gs = U.group_statistics(pooled,
                        metrics=('bos_logit', 'comp_lse_logit', 'margin_lse', 'entropy'))
print('ICC(1,1) at the last layer:')
for i, name in enumerate(gs.metric_names):
    print(f'  {name:<16} {float(gs.icc[-1, i]):+.3f}')
gs.save(EXP_DIR / 'analysis' / 'group_statistics.pt')
fig_gs = U.plot_group_statistics(gs, EXP_DIR / 'figures' / 'analysis_group_statistics.png', show=True)

# save the other three analyses as reusable tensors too, and a JSON summary
_ext, _figs = U.run_extended_analyses(
    pooled, EXP_DIR,
    group_metrics=('bos_logit', 'comp_lse_logit', 'margin_lse', 'entropy'),
    make_figures=False)
print('\\nsaved reusable tensors to', EXP_DIR / 'analysis')
for p in sorted((EXP_DIR / 'analysis').glob('*')):
    print('  ', p.name)

---
## Part 13 · Hand-off to Phases 3.2 and 3.3

Within a KV group every head sees the **same** `k_sink`, so the sink logit
collapses to a projection of that head's query onto one shared direction:

$$\ell^{h}_{t,0} \;=\; \frac{\lVert k_0^{g}\rVert}{\sqrt{D}}\,
\bigl\langle q^{h}_t,\; \hat{n}_t \bigr\rangle,
\qquad \hat n_t = k_0^{g}/\lVert k_0^{g}\rVert$$

and the difference between two heads in the same group is **exactly**

$$\ell^{h}_{t,0} - \ell^{h'}_{t,0}
= \frac{1}{\sqrt D}\bigl\langle q^{h}_t - q^{h'}_t,\; k_0^{g}\bigr\rangle$$

with no residual term — the model itself holds `K` fixed, which is what makes
the GQA angle a natural experiment rather than a correlation.

For **Phase 3.3**, note that `W_Q` enters the forward pass only through
`q = W_Q x`. Substituting stored queries is therefore *identical* to swapping the
weight matrix — same function, no checkpoint surgery, fully reversible. Keys were
saved pre-`repeat_kv`, so `cap.keys[:, :, 0, :]` **is** the shared sink key with
no deduplication needed.

The cell below previews both, loading only from disk.

In [ ]:
# --- Manifest + hand-off preview (artefacts only, no inference) --------------
U.write_manifest(EXP_DIR, dict(
    phase='3.1', model=CONFIG['MODEL_NAME'], device=DEVICE, config=CONFIG,
    benchmark=dict(path=str(bm['path']), fingerprint=bm['fingerprint'],
                   n_pairs=len(bm['pairs']), warnings=bm['warnings'],
                   prompt_mode=CONFIG['PROMPT_MODE'],
                   languages=list(CONFIG['LANGUAGES'])),
    sequences=manifest_seqs, validation=val_summary, pooled_rows=len(pooled),
    saved_full_matrices=CONFIG['SAVE_FULL_MATRICES'],
    figures={k: str(v) for k, v in figs.items()},
))

# --- 3.2a: within-KV-group spread of mean sink margin, straight from the table
print('within-KV-group spread of mean LSE margin (nats)')
print(f"  {'layer':>5}  {'mean spread':>11}  {'worst group':>11}  {'spread':>7}")
for Lx in torch.unique(pooled['layer']).tolist():
    sub = pooled.where(layer=Lx)
    per_head = {h: (float(sub.where(head=h)['margin_lse'].mean()),
                    int(sub.where(head=h)['kv_group'][0]))
                for h in torch.unique(sub['head']).tolist()}
    groups = {}
    for h, (m, g) in per_head.items():
        groups.setdefault(g, []).append(m)
    sp = {g: max(v) - min(v) for g, v in groups.items() if len(v) > 1}
    if sp:
        w = max(sp, key=sp.get)
        print(f'  {Lx:>5}  {sum(sp.values())/len(sp):>11.4f}  {w:>11}  {sp[w]:>7.4f}')

# --- 3.2b: the geometry, from the stored vectors
cap0 = U.load_capture(EXP_DIR / 'captures' / 'seq000.pt')
q = cap0.queries.double()                              # [L,H,T,D] post-RoPE
k_sink = cap0.keys.double()[:, :, 0, :]                # [L,G,D]   the shared sink key
k_h = k_sink.index_select(1, cap0.kv_group_of_head)    # [L,H,D]
num = torch.einsum('lhtd,lhd->lht', q, k_h)
cos = num / (q.norm(dim=-1) * k_h.norm(dim=-1).unsqueeze(-1)).clamp_min(1e-12)
print(f"\ncos(q_t^h, k_sink): shape {tuple(cos.shape)}  "
      f"range [{cos.min():+.3f}, {cos.max():+.3f}]")

n_rep, G_ = cap0.n_rep, cap0.keys.shape[1]
mean_cos = cos.mean(dim=-1)
gaps = torch.stack([mean_cos[:, g*n_rep:(g+1)*n_rep].max(1).values
                    - mean_cos[:, g*n_rep:(g+1)*n_rep].min(1).values
                    for g in range(G_)], dim=1)
print(f'within-group cosine gap: mean {gaps.mean():.4f}, max {gaps.max():.4f}')
print('\nPhase 3.3 sketch -- swapping head h queries for head h\' is exactly a W_Q swap:')
print("    q_alt = cap.queries.clone(); q_alt[:, h] = cap.queries[:, hp]")
print("    new_logit = (q_alt[:, h] * k_sink[g(h)]).sum(-1) * cap.meta['scaling']")

---
## Part 14 · Download

Bundles the figures, metric tables, and the extended-analysis tensors (including
the Phase 3.2 group-statistics hand-off). **Captures are excluded** — they are
the bulk of the payload and already persist on Drive for Phases 3.2 / 3.3, which
read them in place rather than through a download.

In [ ]:
# --- Download this experiment's lightweight outputs --------------------------
import shutil, tempfile
stage = Path(tempfile.mkdtemp()) / 'experiment3.1'
(stage / 'figures').mkdir(parents=True)
(stage / 'metrics').mkdir(parents=True)
(stage / 'analysis').mkdir(parents=True)
for p in (EXP_DIR / 'figures').glob('*.png'):
    shutil.copy2(p, stage / 'figures' / p.name)
for p in (EXP_DIR / 'metrics').glob('*'):
    shutil.copy2(p, stage / 'metrics' / p.name)
for p in (EXP_DIR / 'analysis').glob('*'):
    shutil.copy2(p, stage / 'analysis' / p.name)
for p in EXP_DIR.glob('*.json'):
    shutil.copy2(p, stage / p.name)

zp = shutil.make_archive(str(Path('/content' if IN_COLAB else '.') / 'phase31_outputs'),
                         'zip', stage)
print('Bundled:', zp, f'({Path(zp).stat().st_size/1024**2:.1f} MB)')
print('Captures left on Drive at:', EXP_DIR / 'captures')
if IN_COLAB:
    from google.colab import files
    files.download(zp)